In [23]:
# Change your team name to Group_xx
TEAM_NAME = 'Group_01'

In [24]:
# NOTE: you CAN change this cell
# If you want to use your own database, download it here
# !gdown ...

In [25]:
# NOTE: you CAN change this cell
# Add more to your needs
# you must place ALL pip install here
!pip install editdistance

In [26]:
# NOTE: you CAN change this cell
# import your library here
import time

In [27]:
# NOTE: you MUST change this cell
# New methods / functions must be written under class Solution.
class Solution:
    def __init__(
        self,
        list_province: list[str],
        list_ward: list[str],
        list_street: list[str]
    ): # list province, ward, street
        self.list_province = list_province
        self.list_ward = list_ward
        self.list_street = list_street

        # write your preprocess here, add more method if needed
        pass

    def process(self, s: str):
        # write your process string here
        return {
            "province": self.list_province[0],
            "ward": self.list_ward[0],
            "street": self.list_street[0],
        }

In [28]:
# Download public test
!rm -rf test.json
!gdown --fuzzy https://drive.google.com/file/d/18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW/view?usp=sharing -O test.json
!gdown --fuzzy https://drive.google.com/file/d/1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU/view?usp=drive_link -O list_province.txt
!gdown --fuzzy https://drive.google.com/file/d/1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h/view?usp=drive_link -O list_ward.txt
!gdown --fuzzy https://drive.google.com/file/d/1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa/view?usp=drive_link -O list_street.txt

Downloading...
From: https://drive.google.com/uc?id=18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW
To: /content/test.json
100% 42.7k/42.7k [00:00<00:00, 72.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU
To: /content/list_province.txt
100% 392/392 [00:00<00:00, 1.27MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h
To: /content/list_ward.txt
100% 1.65k/1.65k [00:00<00:00, 5.73MB/s]
Downloading...
From: https://drive.google.com/uc?id=1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa
To: /content/list_street.txt
100% 6.65k/6.65k [00:00<00:00, 20.1MB/s]


In [29]:
def _judge_normalize(s: str, context=None):
    return s

EXCEL_FILE = f'{TEAM_NAME}.xlsx'

import json
import time
with open('test.json') as f:
    data = json.load(f)

with open("list_province.txt") as f:
    list_province = f.read().strip().splitlines()

with open("list_ward.txt") as f:
    list_ward = f.read().strip().splitlines()

with open("list_street.txt") as f:
    list_street = f.read().strip().splitlines()

summary_only = True
df = []
solution = Solution(
    list_province=list_province,
    list_ward=list_ward,
    list_street=list_street,
)
timer = []
correct = 0
for test_idx, data_point in enumerate(data):
    address = data_point["address"]

    ok = 0
    try:
        answer = data_point["result"]
        answer["province_normalized"] = _judge_normalize(answer["province"])
        answer["ward_normalized"] = _judge_normalize(answer["ward"])
        answer["street_normalized"] = _judge_normalize(answer["street"])

        start = time.perf_counter_ns()
        result = solution.process(address)
        finish = time.perf_counter_ns()
        timer.append(finish - start)
        result["province_normalized"] = _judge_normalize(result["province"])
        result["ward_normalized"] = _judge_normalize(result["ward"])
        result["street_normalized"] = _judge_normalize(result["street"])

        province_correct = int(answer["province_normalized"] == result["province_normalized"])
        ward_correct = int(answer["ward_normalized"] == result["ward_normalized"])
        street_correct = int(answer["street_normalized"] == result["street_normalized"])
        ok = province_correct + street_correct + ward_correct

        df.append([
            test_idx,
            address,

            answer["province"],
            result["province"],
            answer["province_normalized"],
            result["province_normalized"],
            province_correct,

            answer["ward"],
            result["ward"],
            answer["ward_normalized"],
            result["ward_normalized"],
            ward_correct,

            answer["street"],
            result["street"],
            answer["street_normalized"],
            result["street_normalized"],
            street_correct,

            ok,
            timer[-1] / 1_000_000_000,
        ])
    except Exception as e:
        print(f"{answer = }")
        print(f"{result = }")
        df.append([
            test_idx,
            address,

            answer["province"],
            "EXCEPTION",
            answer["province_normalized"],
            "EXCEPTION",
            0,

            answer["ward"],
            "EXCEPTION",
            answer["ward_normalized"],
            "EXCEPTION",
            0,

            answer["street"],
            "EXCEPTION",
            answer["street_normalized"],
            "EXCEPTION",
            0,

            0,
            0,
        ])
        # any failure count as a zero correct
        pass
    correct += ok


    if not summary_only:
        # responsive stuff
        print(f"Test {test_idx:5d}/{len(data):5d}")
        print(f"Correct: {ok}/3")
        print(f"Time Executed: {timer[-1] / 1_000_000_000:.4f}")


print(f"-"*30)
total = len(data) * 3
score_scale_10 = round(correct / total * 10, 2)
if len(timer) == 0:
    timer = [0]
max_time_sec = round(max(timer) / 1_000_000_000, 4)
avg_time_sec = round((sum(timer) / len(timer)) / 1_000_000_000, 4)

import pandas as pd

df2 = pd.DataFrame(
    [[correct, total, score_scale_10, max_time_sec, avg_time_sec]],
    columns=['correct', 'total', 'score / 10', 'max_time_sec', 'avg_time_sec',],
)

columns = [
    'ID',
    'text',

    'province',
    'province_student',
    'province_normalized',
    'province_student_normalized',
    'province_correct',

    'ward',
    'ward_student',
    'ward_normalized',
    'ward_student_normalized',
    'ward_correct',

    'street',
    'street_student',
    'street_normalized',
    'street_student_normalized',
    'street_correct',

    'total_correct',
    'time_sec',
]

df = pd.DataFrame(df)
df.columns = columns

print(f'{TEAM_NAME = }')
print(f'{EXCEL_FILE = }')
print(df2)

!pip install xlsxwriter
writer = pd.ExcelWriter(EXCEL_FILE, engine='xlsxwriter')
df2.to_excel(writer, index=False, sheet_name='summary')
df.to_excel(writer, index=False, sheet_name='details')
writer.close()

self.list_province = ['An Giang', 'Bắc Ninh', 'Cà Mau', 'Cần Thơ', 'Gia Lai', 'Huế', 'Hà Nội', 'Hà Tĩnh', 'Hưng Yên', 'Hải Phòng', 'Hồ Chí Minh', 'Khánh Hòa', 'Lai Châu', 'Lào Cai', 'Lâm Đồng', 'Lạng Sơn', 'Nghệ An', 'Ninh Bình', 'Phú Thọ', 'Quảng Ngãi', 'Quảng Ninh', 'Quảng Trị', 'Sơn La', 'Thanh Hóa', 'Thái Nguyên', 'Tuyên Quang', 'Tây Ninh', 'Vĩnh Long', 'Điện Biên', 'Đà Nẵng', 'Đồng Nai', 'Đồng Tháp']
self.list_ward = ['An Bình', 'An Hải', 'An Hội', 'An Lão', 'Ba Đình', 'Bãi Cháy', 'Bình Dương', 'Bình Lư', 'Bình Phước', 'Bình Thủy', 'Bình Đức', 'Bạc Liêu', 'Bắc Cam Ranh', 'Bắc Giang', 'Bắc Hồng Lĩnh', 'Bắc Kạn', 'Bắc Nha Trang', 'Bến Cát', 'Bồ Đề', 'Cam Đường', 'Cao Xanh', 'Chánh Hiệp', 'Chí Linh', 'Cái Khế', 'Cẩm Lệ', 'Cẩm Thành', 'Cửa Lò', 'Duy Hà', 'Dầu Tiếng', 'Giảng Võ', 'Hoàn Kiếm', 'Hà Giang 1', 'Hà Giang 2', 'Hà Lầm', 'Hà Nam', 'Hàm Rồng', 'Hòa Bình', 'Hòa Cường', 'Hòa Khánh', 'Hương An', 'Hạ Long', 'Hạc Thành', 'Hải Châu', 'Hải Vân', 'Hồng Châu', 'Hồng Gai', 'Hồng Hà', 'Ki